Extraindo PDF

In [1]:
!pip install transformers datasets pypdf2 accelerate sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
from pathlib import Path
from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            pages.append(page_text.strip())
    return "\n\n".join(pages)

# Substitua pelo caminho do seu PDF
pdf_path = "20260018701.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 240910

--- INÍCIO DO TEXTO ---

GuiadePlantas
Medicinaisde
Florianópolis

Estacartilhaéfrutodaexperiênciaedotrabalhodemuitas
pessoas.SuapublicaçãofoiviabilizadapeloProjeto“Capaci-
taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do
MinistériodaSaúdede 2 0 1 3.Contribuíramparaseunasci-
mentoprofissionaisegestoresdaredemunicipaldesaúde
(SUSFlorianópolis);professores,técnicoseestudantesdaUni-
versidadeFederaldeSantaCatarina(UFSC);apoiadoresdo
HortoDidáticodePlantasMedicinaisdoHospitalUniversitário
daUFSCesobretudomoradoresdasc


Dividindo em chunks

In [9]:
def dividir_em_paragrafos(texto):
    """Divide o texto em parágrafos usando linhas em branco como separador."""
    paragrafos = texto.split("\n\n")
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]

def chunk_text(paragrafos, max_chunk_chars=1000):
    """Agrupa parágrafos em chunks respeitando o tamanho máximo."""
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos: # Itera sobre cada parágrafo
        if len(chunk_atual) + len(paragrafo) + 2 <= max_chunk_chars: # Verifica se o parágrafo cabe no chunk atual
            chunk_atual += ("\n\n" if chunk_atual else "") + paragrafo # Adiciona o parágrafo ao chunk atual
        else:
            if chunk_atual: # Se o chunk atual não estiver vazio, adiciona-o à lista de chunks
                chunks.append(chunk_atual) # Adiciona o chunk atual à lista de chunks
            chunk_atual = paragrafo # Inicia um novo chunk com o parágrafo atual

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks

In [10]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 154


In [11]:
chunks = chunk_text(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")

Número de chunks: 154

Exemplo de chunk (primeiro):
Estacartilhaéfrutodaexperiênciaedotrabalhodemuitas
pessoas.SuapublicaçãofoiviabilizadapeloProjeto“Capaci-
taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do
MinistériodaSaúdede 2 0 1 3.Contribuíramparaseunasci-
mentoprofissionaisegestoresdaredemunicipaldesaúde
(SUSFlorianópolis);professores,técnicoseestudantesdaUni-
versidadeFederaldeSantaCatarina(UFSC);apoiadoresdo
HortoDidáticodePlantasMedicinaisdoHospitalUniversitário
daUFSCesobretudomoradoresdascomunidadesdeFlori-
anópolis,quehámuitosséculosmantémvivaatradiçãoances-
traldacurapelasplantas–compartilhandosuavaliosa
sabedoriacomtodosetoda...


Carregando modelo

In [5]:
# --- Modelo Seq2Seq ---
seq2seq_id = "google/flan-t5-base" 
seq2seq_tokenizer = AutoTokenizer.from_pretrained(seq2seq_id)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_id)

generator_seq2seq = pipeline(
    "text-generation",
    model=seq2seq_model,
    tokenizer=seq2seq_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3
)

print("Modelos carregados com sucesso.")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1529.78it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

Modelos carregados com sucesso.
